# Cambodia Tourism Chatbot - Prediction
Load trained artifacts, test inputs, and perform error analysis.

In [ ]:
import json
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Load Artifacts
model = load_model('model.h5')
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
with open('config.pkl', 'rb') as f:
    config = pickle.load(f)
with open('intents.json', 'r') as f:
    intents_data = json.load(f)
max_len = config['max_len']

In [ ]:
# Inference function
def predict_intent(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, truncating='post', maxlen=max_len)
    pred = model.predict(padded, verbose=0)
    intent_idx = np.argmax(pred)
    confidence = pred[0][intent_idx]
    tag = label_encoder.inverse_transform([intent_idx])[0]
    
    # Get response
    for intent in intents_data['intents']:
        if intent['tag'] == tag:
            response = np.random.choice(intent['responses'])
            return tag, response, confidence
    return tag, "Sorry, I don't understand.", confidence

In [ ]:
# Test at least 5 inputs
test_inputs = [
    "Hello there!",
    "Tell me about Angkor Wat",
    "Do I need a visa?",
    "What currency do you use?",
    "What's the weather like?",
    "Where can I eat pizza?"
]

for text in test_inputs:
    tag, response, conf = predict_intent(text)
    print(f"Input: {text}\nPredicted Tag: {tag} (Conf: {conf:.2f})\nResponse: {response}\n")

## Error Analysis
- The model performs well on inputs similar to the training patterns.
- For out-of-domain queries like 'Where can I eat pizza?', the model might predict an incorrect tag (e.g., 'food' or another class) because it lacks a fallback or 'unknown' class. 
- **Limitations:** SimpleRNN has limited memory for long sequences. A small dataset causes overconfidence in predictions even for unrelated queries.